# 📖 Story Generation Pipeline — Google Colab

**Four-stage pipeline:** User Prompt → MoPS Premise → DOME Memory → T5 Outline → BART Story

| Stage | Module | Model |
|-------|--------|-------|
| 1 | Premise expansion (MoPS-inspired) | Template-based |
| 2 | Memory extraction (DOME-inspired) | spaCy NER |
| 3 | Outline generation (EtriCA-inspired) | T5-small |
| 4 | Story generation (Hierarchical) | BART-base |

### Notebook sections
1. ⚙️ Setup & Dependencies
2. 📁 Google Drive Mount & Paths
3. 📊 Data Preparation (ROCStories + WritingPrompts)
4. 🏋️ Training (ROCStories + WritingPrompts)
5. 📏 Comprehensive Evaluation (ROUGE-L / BLEU / METEOR / BERTScore)
6. 🔬 Ablation Study (6 conditions)
7. 🎨 Interactive Story Generation Demo

---
> **Before you start:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB")
else:
    print("⚠️  No GPU detected!")
    print("   Go to Runtime → Change runtime type → GPU (T4)")

---
## ⚙️ 1. Setup & Dependencies

In [ ]:
# ── Install all required packages ─────────────────────────────────────────
# Run once per Colab session (takes ~2 min)
!pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu128
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.30.0 \
    sentencepiece>=0.2.0 \
    rouge-score>=0.1.2 \
    sacrebleu>=2.4.0 \
    nltk>=3.8.0 \
    bert-score>=0.3.13 \
    spacy>=3.7.0 \
    datasets>=2.19.0

!python -m spacy download en_core_web_sm -q

import nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

print("✅ All dependencies installed")

---
## 📁 2. Google Drive Mount & Paths

Models and processed data are saved to Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# ── Configure this path to your preferred Drive folder ─────────────────────
DRIVE_DIR = "/content/drive/MyDrive/StoryGeneration"

os.makedirs(DRIVE_DIR, exist_ok=True)
os.chdir(DRIVE_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Upload / verify project source files ──────────────────────────────────
#
# Option A (recommended): upload the whole repo zip once and unzip:
#   from google.colab import files
#   files.upload()                         # upload StoryGeneration.zip
#   !unzip -q StoryGeneration.zip
#
# Option B: clone from GitHub:
#   !git clone https://github.com/YOUR_USERNAME/StoryGeneration.git .
#
# After uploading, run this cell to verify:
import pathlib

REQUIRED = [
    "inference.py", "metrics.py", "evaluate.py", "ablation.py",
    "train_outline.py", "train_story.py",
    "prepare_data.py", "prepare_writingprompts.py",
]

all_ok = True
for fname in REQUIRED:
    status = "✅" if pathlib.Path(fname).exists() else "❌ MISSING"
    print(f"  {fname:<35} {status}")
    if "MISSING" in status:
        all_ok = False

print()
print("✅ All source files present" if all_ok else "❌ Upload missing files before continuing")

---
## 📊 3. Data Preparation

### 3a. ROCStories

Upload `rocstories_train.txt` and `rocstories_test.txt` to `data/raw/`.
These files use a tab-separated format: `id<TAB>sentence1. sentence2. ... sentence5.`

In [ ]:
import os, pathlib

os.makedirs("data/raw", exist_ok=True)

train_ok = pathlib.Path("data/raw/rocstories_train.txt").exists()
test_ok  = pathlib.Path("data/raw/rocstories_test.txt").exists()

if train_ok and test_ok:
    print("✅ ROCStories files already present — skipping upload")
else:
    print("Uploading ROCStories files ...")
    from google.colab import files
    uploaded = files.upload()   # select both .txt files
    for fname, data in uploaded.items():
        dest = pathlib.Path("data/raw") / fname
        dest.write_bytes(data)
        print(f"  Saved {dest}")

In [ ]:
# ── Prepare ROCStories JSONL files ────────────────────────────────────────
# Produces: data/processed/outline_{train,val,test}.jsonl
#           data/processed/story_{train,val,test}.jsonl

if pathlib.Path("data/processed/story_train.jsonl").exists():
    print("✅ Processed ROCStories data already exists — skipping")
else:
    !python prepare_data.py

### 3b. WritingPrompts *(optional — for WP fine-tuning)*

Downloads `euclaise/writingprompts` from HuggingFace (~272K stories) and creates
filtered JSONL files. Use `--max-stories` to limit size for faster experiments.

In [ ]:
# ── Prepare WritingPrompts JSONL files ────────────────────────────────────
# Full dataset (~272K): takes ~20 min
# Subset (50K):  takes ~4 min — good for experiments

MAX_WP_STORIES = 50000   # set to None for the full dataset

if pathlib.Path("data/processed/wp_story_train.jsonl").exists():
    print("✅ Processed WritingPrompts data already exists — skipping")
else:
    flag = f"--max-stories {MAX_WP_STORIES}" if MAX_WP_STORIES else ""
    !python prepare_writingprompts.py {flag}

---
## 🏋️ 4. Training

### 4a. Outline Generator — T5-small (ROCStories)

Trains T5-small to map `story title → event1 | event2 | event3`.
~15 min / epoch on T4 GPU.

In [ ]:
# ── Train T5 outline model on ROCStories ──────────────────────────────────
# First session: train from scratch
!python train_outline.py --data roc --epochs 3

In [ ]:
# ── Resume T5 outline training (run in subsequent Colab sessions) ──────────
!python train_outline.py --data roc --epochs 1 --resume

### 4b. Story Generator — BART-base (ROCStories)

Trains BART-base to map `title outline: [events] [MEM] ... → full story`.
~25 min / epoch on T4 GPU.

In [ ]:
# ── Train BART story model on ROCStories ──────────────────────────────────
!python train_story.py --data roc --epochs 3

In [ ]:
# ── Resume BART story training ─────────────────────────────────────────────
!python train_story.py --data roc --epochs 1 --resume

### 4c. WritingPrompts Fine-tuning *(optional)*

Re-trains both models on filtered WritingPrompts data.
Gradient accumulation (`--grad-accum 4`) keeps effective batch size at 32
despite WP's longer sequences.

In [ ]:
# ── Train T5 outline model on WritingPrompts ──────────────────────────────
# Session 1 (first run)
!python train_outline.py --data wp --epochs 1 --grad-accum 4

In [ ]:
# ── Resume WP outline training (subsequent sessions) ──────────────────────
!python train_outline.py --data wp --epochs 1 --grad-accum 4 --resume

In [ ]:
# ── Train BART story model on WritingPrompts ───────────────────────────────
# Session 1 (first run)
!python train_story.py --data wp --epochs 1 --grad-accum 4

In [ ]:
# ── Resume WP story training (subsequent sessions) ────────────────────────
!python train_story.py --data wp --epochs 1 --grad-accum 4 --resume

---
## 📏 5. Comprehensive Evaluation

Evaluates the full pipeline on four metrics:

| Metric | What it measures |
|--------|------------------|
| **ROUGE-L** | Longest common subsequence overlap |
| **BLEU** | n-gram precision (sacrebleu) |
| **METEOR** | Harmonic mean of precision & recall with stemming & synonyms |
| **BERTScore** | Semantic similarity via contextual embeddings |

> 💡 Low BLEU/ROUGE is expected — creative stories are very diverse.

In [ ]:
# ── Evaluate ROCStories pipeline — 500 validation examples ────────────────
!python evaluate.py --data roc --n 500 --split val

In [ ]:
# ── Evaluate on test split (final numbers) ────────────────────────────────
!python evaluate.py --data roc --n 500 --split test

In [ ]:
# ── Evaluate WritingPrompts pipeline ─────────────────────────────────────
# (Only run if you trained WP models in Section 4c)
!python evaluate.py --data wp --n 200 --split val

In [ ]:
# ── Python API — evaluate and inspect results programmatically ────────────
import evaluate as eval_module

results = eval_module.evaluate(
    data="roc",
    split="val",
    n=100,                          # smaller n for a quick check
    use_memory=True,
    bert_model="distilbert-base-uncased",
)

print("\nReturned dict:", results)

---
## 🔬 6. Ablation Study

Tests **6 conditions** to measure the contribution of each pipeline component:

| Condition | Description |
|-----------|-------------|
| `full` | **Baseline** — T5 event outline + DOME memory (ROC models) |
| `no_memory` | Remove DOME memory module |
| `no_outline` | Remove outline — title → BART directly |
| `with_premise` | Add structured premise text to BART input |
| `sent_outline` | Oracle sentence outline (upper bound on outline quality) |
| `wp_models` | Full pipeline with WritingPrompts-trained models |

The `sent_outline` condition uses actual sentences from the **reference story**
as the outline, showing the ceiling gain from a perfect outline generator.

In [ ]:
# ── Quick ablation smoke test (50 examples, ~10 min on T4) ────────────────
!python ablation.py --n 50 --split val

In [ ]:
# ── Full ablation (200 examples per condition, ~40 min on T4) ─────────────
!python ablation.py --n 200 --split val

In [ ]:
# ── Run only a subset of conditions ───────────────────────────────────────
!python ablation.py --n 200 --conditions full no_memory no_outline

In [ ]:
# ── Python API — run ablation and get results as a dict ───────────────────
import importlib, ablation as ab
importlib.reload(ab)

results = ab.ablation(
    n=50,
    split="val",
    bert_model="distilbert-base-uncased",
    conditions=["full", "no_memory", "no_outline"],
)

# ── Pretty-print as a pandas DataFrame ───────────────────────────────────
import pandas as pd

df = pd.DataFrame(results).T
df.index.name = "condition"
df = df[["rouge_l", "bleu", "meteor", "bertscore"]]
df.columns = ["ROUGE-L", "BLEU", "METEOR", "BERTScore"]
df = df.round(4)

# Highlight best per column
display(df.style.highlight_max(axis=0, color="#d4f0d4"))

---
## 🎨 7. Interactive Story Generation Demo

Generate stories from your own prompts and inspect each pipeline stage.

In [ ]:
import inference

# ── Choose which model to use ─────────────────────────────────────────────
# 'roc' = ROCStories models  |  'wp' = WritingPrompts models
MODEL = "roc"

if MODEL == "wp":
    inference.T5_CHECKPOINT   = "models/t5_outline_wp"
    inference.BART_CHECKPOINT = "models/bart_story_wp"
else:
    inference.T5_CHECKPOINT   = "models/t5_outline"
    inference.BART_CHECKPOINT = "models/bart_story"

# Clear model cache so the new checkpoints are loaded
inference._outline_cache = None
inference._story_cache   = None

# ── Your story prompt ────────────────────────────────────────────────────
PROMPT = "She finally found what she had been looking for"

result = inference.run_pipeline(PROMPT)

In [ ]:
# ── Compare pipeline variants on the same prompt ─────────────────────────
import inference
from inference import generate_outline, generate_story, extract_memory, expand_premise

PROMPT = "The last train left without him"

# Pre-compute shared inputs
outline = generate_outline(PROMPT)
memory  = extract_memory(PROMPT)
premise = expand_premise(PROMPT)

print(f"Prompt  : {PROMPT}")
print(f"Outline : {outline}")
print(f"Memory  : {memory or '(none)'}")
print()

variants = {
    "Full pipeline"    : dict(use_memory=True,  use_outline=True,  use_premise=False),
    "No memory"        : dict(use_memory=False, use_outline=True,  use_premise=False),
    "No outline"       : dict(use_memory=False, use_outline=False, use_premise=False),
    "With premise"     : dict(use_memory=True,  use_outline=True,  use_premise=True),
}

for label, flags in variants.items():
    story = generate_story(PROMPT, outline, memory, premise=premise, **flags)
    print(f"[{label}]")
    print(story)
    print()

In [ ]:
# ── Batch generation — run on a list of prompts ───────────────────────────
import inference
from inference import generate_outline, generate_story, extract_memory

inference._outline_cache = None
inference._story_cache   = None

prompts = [
    "She finally found what she had been looking for",
    "The dog had been waiting at the door for three days",
    "He opened the letter and his hands began to shake",
    "The old library held a secret no one had discovered",
    "They said it was impossible, but she proved them wrong",
]

stories = []
for prompt in prompts:
    outline = generate_outline(prompt)
    memory  = extract_memory(prompt)
    story   = generate_story(prompt, outline, memory)
    stories.append({"prompt": prompt, "outline": outline, "story": story})
    print(f"✓ {prompt[:50]}")

print(f"\nGenerated {len(stories)} stories.")

# Pretty print the last one as a sample
s = stories[-1]
print(f"\n--- Sample output ---")
print(f"Prompt  : {s['prompt']}")
print(f"Outline : {s['outline']}")
print(f"Story   : {s['story']}")

In [ ]:
# ── Evaluate a single generated story against a reference ─────────────────
import metrics

hypothesis = "She searched for years until she stumbled upon a small shop. Inside was the necklace her grandmother had lost. The shopkeeper smiled as if he had been expecting her. She paid with the last of her savings and walked home in tears. Some things are worth any price."
reference  = "She had been looking for the missing heirloom her whole life. One afternoon she found it in an antique market downtown. The seller told her it had been there for twenty years. She could hardly believe her eyes when she saw the initials on the back. She brought it home and placed it on the mantle where it belonged."

scores = metrics.compute_all([hypothesis], [reference], verbose=True)
print("\nScores:", scores)

---
## 💾 8. Save Results to Drive

Save evaluation and ablation results as JSON/CSV for your paper.

In [ ]:
import json, pathlib, pandas as pd
import importlib

# ── Run full evaluation ────────────────────────────────────────────────────
import evaluate as eval_mod
importlib.reload(eval_mod)

roc_scores = eval_mod.evaluate(data="roc", split="val", n=500)

# ── Run full ablation ──────────────────────────────────────────────────────
import ablation as abl_mod
importlib.reload(abl_mod)

ablation_scores = abl_mod.ablation(n=200, split="val")

# ── Save to Drive ─────────────────────────────────────────────────────────
results_dir = pathlib.Path("results")
results_dir.mkdir(exist_ok=True)

# JSON
(results_dir / "evaluation_roc.json").write_text(
    json.dumps(roc_scores, indent=2))
(results_dir / "ablation.json").write_text(
    json.dumps(ablation_scores, indent=2))

# CSV
df_eval = pd.DataFrame([roc_scores], index=["ROCStories"])
df_eval.to_csv(results_dir / "evaluation_roc.csv")

df_abl = pd.DataFrame(ablation_scores).T
df_abl.to_csv(results_dir / "ablation.csv")

print("✅ Results saved to", results_dir.resolve())
print()

# ── Pretty display ────────────────────────────────────────────────────────
print("=== Evaluation ===")
display(df_eval.round(4))

print("\n=== Ablation ===")
display(df_abl.round(4).style.highlight_max(axis=0, color="#d4f0d4"))